In [1]:
import arcpy
import os
arcpy.env.overwriteOutput = True
arcpy.env.outputCoordinateSystem = None
project_folder = r"C:/Users/Chels/OneDrive - University of Illinois - Urbana/Ch5_Sackett_Impacts/Wetland Mapping"
nhd_gdb = os.path.normpath(os.path.join(project_folder, "Illinois_NHD_Shapefiles.gdb"))
jd_water_gdb = os.path.normpath(os.path.join(project_folder, "Jurisdictional_Water_Shapefiles.gdb"))

# Create layer of intermittent and perennial waterbodies

## Remove NHD wetlands

In [2]:
# starting waterbody layer
nhd_waterbody_nad1983 = os.path.normpath(os.path.join(nhd_gdb,"NHDWaterbody_IL_AlbersNAD1983"))

In [3]:
# make field for layer type
arcpy.management.AddField(nhd_waterbody_nad1983,'Type','Text');

In [4]:
arcpy.management.CalculateField(
    in_table=nhd_waterbody_nad1983,   
    field="Type",
    expression="'Waterbody'",
    expression_type="PYTHON3"
);

In [5]:
# nonwetland waterbodies
nhd_waterbody_step0 = os.path.normpath(os.path.join(nhd_gdb,"NHDWaterbody_IL_Step0_NonWetlands"))

In [6]:
arcpy.conversion.ExportFeatures(
    in_features = nhd_waterbody_nad1983,
    out_features = nhd_waterbody_step0,
    where_clause = "fcode NOT IN (46600, 46601, 46602)"
);

In [7]:
count = int(arcpy.management.GetCount(nhd_waterbody_nad1983)[0])
count

164412

In [8]:
count0 = int(arcpy.management.GetCount(nhd_waterbody_step0)[0])
count0

154562

## Isolate lakes, ponds, and impoundments

In [9]:
nhd_waterbody_step1 = os.path.normpath(os.path.join(nhd_gdb,"NHDWaterbody_IL_Step1_LakesPondsImpoundments"))

In [10]:
arcpy.conversion.ExportFeatures(
    in_features = nhd_waterbody_step0,
    out_features = nhd_waterbody_step1,
    where_clause = "ftype IN (390) OR fcode IN (43600, 43614, 43615)"
);

In [11]:
count1 = int(arcpy.management.GetCount(nhd_waterbody_step1)[0])
count1

151208

## Create hydrologic field class and assign to perennial or intermittent

In [12]:
arcpy.management.AddField(nhd_waterbody_step1, "Final_NHD_Hydro_Class", "LONG");

In [13]:
arcpy.management.AddField(nhd_waterbody_step1, "Brinkerhoff_Hydro_Class", "LONG");

In [14]:
# perennial lakes and reservoirs -> 1
arcpy.management.MakeFeatureLayer(nhd_waterbody_step1,"temp_layer");
arcpy.management.SelectLayerByAttribute(
    in_layer_or_view="temp_layer",
    where_clause="(fcode IN (39004, 39009, 39010, 39011, 39012, 39012, 43615)) OR (fcode = 43600 AND gnis_name IS NOT NULL) OR (fcode = 39000 AND areasqkm > 0.01)",
    selection_type="NEW_SELECTION"
);
count = int(arcpy.management.GetCount("temp_layer")[0])
print(f"Selected: {count} features")
arcpy.management.CalculateField(
    in_table="temp_layer",   
    field="Final_NHD_Hydro_Class",
    expression="1",
    expression_type="PYTHON3"
);
arcpy.management.CalculateField(
    in_table="temp_layer", 
    field="Brinkerhoff_Hydro_Class",
    expression="1",
    expression_type="PYTHON3"
);
arcpy.management.Delete("temp_layer");

Selected: 143178 features


In [15]:
# intermittent lakes and reservoirs -> 2
arcpy.management.MakeFeatureLayer(nhd_waterbody_step1, "temp_layer");
arcpy.management.SelectLayerByAttribute(
    in_layer_or_view="temp_layer",
    where_clause="(fcode IN (39001,39005, 39006, 43614)) OR (fcode = 43600 AND gnis_name IS NULL) OR (fcode = 39000 AND areasqkm <= 0.01)",
    selection_type="NEW_SELECTION"
);
count = int(arcpy.management.GetCount("temp_layer")[0])
print(f"Selected: {count} features")
arcpy.management.CalculateField(
    in_table="temp_layer",   
    field="Final_NHD_Hydro_Class",
    expression="2",
    expression_type="PYTHON3"
);
arcpy.management.CalculateField(
    in_table="temp_layer", 
    field="Brinkerhoff_Hydro_Class",
    expression="2",
    expression_type="PYTHON3"
);
arcpy.management.Delete("temp_layer");

Selected: 8030 features


## Export layer with waterbodies assigned to perennial/intermittent

In [16]:
nhd_waterbody_step2 = os.path.normpath(os.path.join(nhd_gdb,"NHDWaterbody_IL_Step2_Classified"))

In [17]:
arcpy.conversion.ExportFeatures(
    in_features = nhd_waterbody_step1,
    out_features = nhd_waterbody_step2,
);

In [18]:
# rule for calculate variable buffer column
variable_buffer_rule = """
def VarBuff(Hydro_Class):
    if Hydro_Class == 1:
        return '20 Meters'
    elif Hydro_Class == 2:
        return '10 Meters'
    elif Hydro_Class == 3:
        return '5 Meters'
"""

In [19]:
# NHD: make variable buffer distance column
arcpy.management.AddField(nhd_waterbody_step2, "NHD_Variable_Buffer", "TEXT");
arcpy.management.AddField(nhd_waterbody_step2, "Brinkerhoff_Variable_Buffer", "TEXT");

In [20]:
# NHD variable buffer
arcpy.management.CalculateField(
    in_table=nhd_waterbody_step2,
    field="NHD_Variable_Buffer",
    expression = "VarBuff(!Final_NHD_Hydro_Class!)",
    expression_type="PYTHON3",
    code_block = variable_buffer_rule
);

In [21]:
# Brinkerhoff variable buffer
arcpy.management.CalculateField(
    in_table=nhd_waterbody_step2,
    field="Brinkerhoff_Variable_Buffer",
    expression = "VarBuff(!Brinkerhoff_Hydro_Class!)",
    expression_type="PYTHON3",
    code_block = variable_buffer_rule
);

## Create flowline layer with only perennial/intermittent according to Brinkerhoff class

In [22]:
step7_nhd = os.path.normpath(os.path.join(nhd_gdb,"NHDFlowline_Step7_RemoveHydroNAs_JoinBrinkerhoff"))
step8_nhd = os.path.normpath(os.path.join(nhd_gdb,"NHDFlowline_Step8_Brinkerhoff_Nonephemeral"))

In [23]:
arcpy.conversion.ExportFeatures(
    in_features = step7_nhd,
    out_features = step8_nhd,
    where_clause = "Brinkerhoff_Hydro_Class NOT IN (3)"
);

## Select lakes/ponds that intersect perennial/intermittent flowlines according to Brinkerhoff class

In [24]:
nhd_waterbody_step3 = os.path.normpath(os.path.join(nhd_gdb,"NHDWaterbody_IL_Step3_IntersectOnly"))

In [25]:
arcpy.management.MakeFeatureLayer(nhd_waterbody_step2, "temp_layer");
arcpy.management.SelectLayerByLocation(
    in_layer="temp_layer",
    overlap_type="INTERSECT",
    select_features=step8_nhd,
    selection_type="NEW_SELECTION"
);
arcpy.conversion.ExportFeatures(
    in_features = "temp_layer",
    out_features = nhd_waterbody_step3,
);

# Create flowline, area, and waterbody buffers

## Fixed distance buffers (1 m, 10 m, and 100 m)

In [26]:
# three fixed buffer distances
fixed_buffers = [1, 10, 100]

In [27]:
arcpy.management.AddField(step7_nhd,'Type','Text');

In [28]:
arcpy.management.CalculateField(
    in_table=step7_nhd,   
    field="Type",
    expression="'Flowline'",
    expression_type="PYTHON3"
);

### Flowlines

In [29]:
for b in fixed_buffers:
    arcpy.analysis.PairwiseBuffer(
        in_features=step7_nhd,
        out_feature_class=os.path.normpath(os.path.join(jd_water_gdb, f"NHDFlowline_Buffer{b}")),
        buffer_distance_or_field=f"{b} Meters",
        dissolve_option="NONE",
        method="PLANAR"
    )
    arcpy.analysis.PairwiseBuffer(
        in_features=step7_nhd,
        out_feature_class=os.path.normpath(os.path.join(jd_water_gdb, f"NHDFlowline_Brinkerhoff_Buffer{b}")),
        buffer_distance_or_field=f"{b} Meters",
        dissolve_option="NONE",
        method="PLANAR"
    )

### Area features

In [30]:
nhd_area_feature_path = os.path.normpath(os.path.join(nhd_gdb, "NHDArea_StreamRiversCanals"))
for b in fixed_buffers:
    arcpy.analysis.PairwiseBuffer(
        in_features=nhd_area_feature_path,
        out_feature_class=os.path.normpath(os.path.join(jd_water_gdb, f"NHDArea_Buffer{b}")),
        buffer_distance_or_field=f"{b} Meters",
        dissolve_option="NONE",
        method="PLANAR"
    )

### Waterbodies

In [31]:
for b in fixed_buffers:
    arcpy.analysis.PairwiseBuffer(
        in_features=nhd_waterbody_step2,
        out_feature_class=os.path.normpath(os.path.join(jd_water_gdb, f"NHDWaterbody_Buffer{b}")),
        buffer_distance_or_field=f"{b} Meters",
        dissolve_option="NONE",
        method="PLANAR"
    )
    arcpy.analysis.PairwiseBuffer(
        in_features=nhd_waterbody_step3,
        out_feature_class=os.path.normpath(os.path.join(jd_water_gdb, f"NHDWaterbody_Brinkerhoff_Buffer{b}")),
        buffer_distance_or_field=f"{b} Meters",
        dissolve_option="NONE",
        method="PLANAR"
    )

## Variable distance buffers (Perennial: 20 m, Intermittent: 10 m, and Ephemeral: 5 m)

In [32]:
# three variable buffer distances
variable_buffers = [20, 10, 5]

### Flowlines

#### Buffers based on original NHD flow permanence categories

In [33]:
# create field
arcpy.management.AddField(step7_nhd, "NHD_Variable_Buffer", "TEXT");

In [34]:
# make variable buffer column
arcpy.management.CalculateField(
    in_table=step7_nhd,
    field="NHD_Variable_Buffer",
    expression = "VarBuff(!Final_NHD_Hydro_Class!)",
    expression_type="PYTHON3",
    code_block = variable_buffer_rule
);

In [35]:
# make buffer files
arcpy.analysis.PairwiseBuffer(
    in_features=step7_nhd,
    out_feature_class=os.path.normpath(os.path.join(jd_water_gdb, "NHDFlowline_Buffer20")),
    buffer_distance_or_field="NHD_Variable_Buffer",
    dissolve_option="NONE",
    method="PLANAR"
);

#### Buffers based on Brinkerhoff-updated flow permanence categories

In [36]:
# create field
arcpy.management.AddField(step7_nhd, "Brinkerhoff_Variable_Buffer", "TEXT");

In [37]:
# make variable buffer column
arcpy.management.CalculateField(
    in_table=step7_nhd,
    field="Brinkerhoff_Variable_Buffer",
    expression = "VarBuff(!Brinkerhoff_Hydro_Class!)",
    expression_type="PYTHON3",
    code_block = variable_buffer_rule
);

In [44]:
# make buffer files
arcpy.analysis.PairwiseBuffer(
    in_features=step7_nhd,
    out_feature_class=os.path.normpath(os.path.join(jd_water_gdb, "NHDFlowline_Brinkerhoff_Buffer20")),
    buffer_distance_or_field="Brinkerhoff_Variable_Buffer",
    dissolve_option="NONE",
    method="PLANAR"
);

### Area features

In [39]:
nhd_area_streams_rivers_canals = os.path.normpath(os.path.join(nhd_gdb, "NHDArea_StreamRiversCanals"))

In [40]:
arcpy.analysis.PairwiseBuffer(
    in_features=nhd_area_streams_rivers_canals,
    out_feature_class=os.path.normpath(os.path.join(jd_water_gdb, "NHDArea_Buffer20")),
    buffer_distance_or_field="NHD_Variable_Buffer",
    dissolve_option="NONE",
    method="PLANAR"
);

### Waterbodies

In [41]:
# make NHD buffer files
arcpy.analysis.PairwiseBuffer(
    in_features=nhd_waterbody_step2,
    out_feature_class=os.path.normpath(os.path.join(jd_water_gdb, "NHDWaterbody_Buffer20")),
    buffer_distance_or_field="NHD_Variable_Buffer",
    dissolve_option="NONE",
    method="PLANAR"
);

In [42]:
# make Brinkerhoff buffer files
arcpy.analysis.PairwiseBuffer(
    in_features=nhd_waterbody_step3,
    out_feature_class=os.path.normpath(os.path.join(jd_water_gdb, "NHDWaterbody_Brinkerhoff_Buffer20")),
    buffer_distance_or_field="Brinkerhoff_Variable_Buffer",
    dissolve_option="NONE",
    method="PLANAR"
);